In [2]:
# Build Salary Band Sheet — basis_master

#Extracts every salary-band factor table (basic + AIDS) from
#`Basis and Assumptions.xlsx` into one long sheet:

#| product | gender | salary_from | inc_adj | acaf_adj |

#Also appends a row to `build_log.csv` for the audit trail.

In [3]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import csv
import polars as pl
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.table import Table, TableStyleInfo

BASE = Path(
    r"J:\Product & Client Solutions\Toolbox\Calculators"
    r"\Business Tools\Tender Summary\GAP Pricing Engine"
)
SOURCE       = BASE / "Basis and Assumptions.xlsx"
MASTER_XLSX  = BASE / "basis_master" / "basis_master_v2026_07.xlsx"
SNAPSHOT_DIR = BASE / "basis_snapshots" / "basis_v2026_07"
BUILD_LOG    = SNAPSHOT_DIR / "build_log.csv"

MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
assert SOURCE.exists()
print("✅ Setup ready")

✅ Setup ready


In [4]:
def find_anchors(ws, text: str) -> list[tuple[int, int]]:
    hits = []
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip() == text:
                hits.append((cell.row, cell.column))
    return hits


def extract_salary_band(ws, anchor_row: int, anchor_col: int,
                        n_factors: int = 1, max_rows: int = 25,
                        max_skip: int = 4
                        ) -> list[tuple[float, ...]]:
    """
    Walk downward from anchor extracting (salary_from, factor1, factor2, ...).

    Robust to sub-header rows: will skip up to `max_skip` non-numeric rows
    before giving up. Stops when we hit a non-numeric row AFTER data has started.
    """
    rows = []
    r = anchor_row + 1
    data_started = False

    # Phase 1: skip up to max_skip non-numeric rows to find the start of data
    for _ in range(max_skip):
        sf = ws.cell(row=r, column=anchor_col).value
        if sf is None:
            r += 1
            continue
        try:
            float(sf)
            break  # found numeric — data starts here
        except (ValueError, TypeError):
            r += 1  # still a header, skip it

    # Phase 2: read data rows until we hit a non-numeric
    for _ in range(max_rows):
        sf = ws.cell(row=r, column=anchor_col).value
        if sf is None:
            break
        try:
            sf_num = float(sf)
        except (ValueError, TypeError):
            break
        factors = []
        for k in range(1, n_factors + 1):
            v = ws.cell(row=r, column=anchor_col + k).value
            factors.append(float(v) if v is not None else None)
        rows.append((sf_num, *factors))
        r += 1

    return rows


def sha256_of(path: Path) -> str:
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

print("✅ Helpers ready (robust to sub-headers)")

✅ Helpers ready (robust to sub-headers)


In [5]:
SPECS = [
    # ===== BASIC SALARY FACTORS =====
    dict(product="gla", gender="M", sheet="GLA Basis",
         anchor="M Sal Band", occurrence=1, n_factors=2),
    dict(product="gla", gender="F", sheet="GLA Basis",
         anchor="F Sal Band", occurrence=1, n_factors=2),

    dict(product="ci", gender="both", sheet="CI Basis",
         anchor="Salary Bands", occurrence=1, n_factors=1),
    dict(product="spouse", gender="both", sheet="Spouse Basis",
         anchor="Salary Bands", occurrence=1, n_factors=1),
    dict(product="adb", gender="M", sheet="ADB Basis",
     anchor="Salary Band", occurrence=2, n_factors=1),
    dict(product="adb", gender="F", sheet="ADB Basis",
     anchor="Salary Band", occurrence=3, n_factors=1),
    dict(product="accdb", gender="both", sheet="AccDB Basis",
         anchor="Salary Bands", occurrence=1, n_factors=1),
    dict(product="dism",  gender="both", sheet="Dismemberment Basis",
         anchor="Salary Bands", occurrence=1, n_factors=1),

    # ===== AIDS SALARY FACTORS =====
    dict(product="ci_aids_acc", gender="both", sheet="CI Basis",
         anchor="Salary Band", occurrence=1, n_factors=1),
    dict(product="spouse_aids", gender="both", sheet="Spouse Basis",
         anchor="Salary Band", occurrence=1, n_factors=1),
    # gla_aids handled in Cell 4 with anchor discovery
]

print(f"✅ {len(SPECS)} standard specs defined (+ LSDB + gla_aids discovery)")

✅ 10 standard specs defined (+ LSDB + gla_aids discovery)


In [6]:
print(f"📂 Opening source: {SOURCE.name}\n")
wb = openpyxl.load_workbook(SOURCE, data_only=True)

all_rows = []
print(f"🔎 Extracting {len(SPECS)} standard tables...\n")

for spec in SPECS:
    ws = wb[spec["sheet"]]
    anchors = find_anchors(ws, spec["anchor"])
    occ = spec["occurrence"] - 1
    if occ >= len(anchors):
        print(f"  ❌ {spec['product']:<15} {spec['gender']:<5} "
              f"'{spec['anchor']}' #{spec['occurrence']} NOT FOUND "
              f"({len(anchors)} matches)")
        continue

    r, c = anchors[occ]
    rows = extract_salary_band(ws, r, c, n_factors=spec["n_factors"])
    print(f"  ✅ {spec['product']:<15} {spec['gender']:<5} "
          f"'{spec['anchor']}' #{spec['occurrence']} → {len(rows):>2} rows")

    for row in rows:
        if spec["n_factors"] == 2:
            sf, inc, acaf = row
            all_rows.append(dict(product=spec["product"], gender=spec["gender"],
                                 salary_from=sf, inc_adj=inc, acaf_adj=acaf))
        else:
            sf, fac = row
            all_rows.append(dict(product=spec["product"], gender=spec["gender"],
                                 salary_from=sf, inc_adj=fac, acaf_adj=None))

# ---------- Duplicate check: CI Basis 'Salary Bands' ----------
print("\n🔎 Verifying CI Basis 'Salary Bands' duplicate is identical...")
ws_ci = wb["CI Basis"]
ci_anchors = find_anchors(ws_ci, "Salary Bands")
if len(ci_anchors) >= 2:
    rows_1 = extract_salary_band(ws_ci, *ci_anchors[0], n_factors=1)
    rows_2 = extract_salary_band(ws_ci, *ci_anchors[1], n_factors=1)
    if rows_1 == rows_2:
        print(f"  ✅ Duplicate confirmed identical ({len(rows_1)} bands) — safely skipping #2")
    else:
        print(f"  ⚠️  Duplicate DIFFERS — investigate!")
else:
    print(f"  ℹ️  Only {len(ci_anchors)} 'Salary Bands' occurrence(s) found")

# ---------- LSDB (side-by-side M/F layout) ----------
# Confirmed layout from source:
#   Sal band | M Band | F Band | M Inc | F Inc | M ACAF | F ACAF
# Offsets from anchor: sf=0, m_inc=3, f_inc=4, m_acaf=5, f_acaf=6
# First data row starts immediately after the header row.
print("\n🔎 Extracting LSDB (side-by-side M+F)...")
ws_lsdb = wb["LSDB Basis"]
lsdb_anchors = find_anchors(ws_lsdb, "Sal band")
if not lsdb_anchors:
    print("  ❌ LSDB 'Sal band' anchor NOT FOUND")
else:
    r0, c0 = lsdb_anchors[0]
    r = r0 + 1  # start immediately after the header row
    lsdb_count = 0
    for _ in range(20):
        sf = ws_lsdb.cell(row=r, column=c0).value
        if sf is None:
            break
        try:
            sf_num = float(sf)
        except (ValueError, TypeError):
            break

        m_inc  = ws_lsdb.cell(row=r, column=c0+3).value
        f_inc  = ws_lsdb.cell(row=r, column=c0+4).value
        m_acaf = ws_lsdb.cell(row=r, column=c0+5).value
        f_acaf = ws_lsdb.cell(row=r, column=c0+6).value

        all_rows.append(dict(product="lsdb", gender="M", salary_from=sf_num,
                             inc_adj=float(m_inc)  if m_inc  is not None else None,
                             acaf_adj=float(m_acaf) if m_acaf is not None else None))
        all_rows.append(dict(product="lsdb", gender="F", salary_from=sf_num,
                             inc_adj=float(f_inc)  if f_inc  is not None else None,
                             acaf_adj=float(f_acaf) if f_acaf is not None else None))
        lsdb_count += 1
        r += 1
    print(f"  ✅ lsdb           M+F   → {lsdb_count} salary bands × 2 = {lsdb_count*2} rows")

df = pl.DataFrame(all_rows).sort(["product", "gender", "salary_from"])
print(f"\n📊 Total rows: {df.height}  |  Distinct products: {df['product'].n_unique()}")
df.group_by(["product", "gender"]).agg(pl.len().alias("n")).sort(["product", "gender"])

📂 Opening source: Basis and Assumptions.xlsx

🔎 Extracting 10 standard tables...

  ✅ gla             M     'M Sal Band' #1 → 10 rows
  ✅ gla             F     'F Sal Band' #1 →  7 rows
  ✅ ci              both  'Salary Bands' #1 → 16 rows
  ✅ spouse          both  'Salary Bands' #1 → 16 rows
  ✅ adb             M     'Salary Band' #2 →  3 rows
  ✅ adb             F     'Salary Band' #3 →  3 rows
  ✅ accdb           both  'Salary Bands' #1 →  3 rows
  ✅ dism            both  'Salary Bands' #1 →  3 rows
  ✅ ci_aids_acc     both  'Salary Band' #1 →  8 rows
  ✅ spouse_aids     both  'Salary Band' #1 →  8 rows

🔎 Verifying CI Basis 'Salary Bands' duplicate is identical...
  ✅ Duplicate confirmed identical (16 bands) — safely skipping #2

🔎 Extracting LSDB (side-by-side M+F)...
  ✅ lsdb           M+F   → 13 salary bands × 2 = 26 rows

📊 Total rows: 103  |  Distinct products: 9


product,gender,n
str,str,u32
"""accdb""","""both""",3
"""adb""","""F""",3
"""adb""","""M""",3
"""ci""","""both""",16
"""ci_aids_acc""","""both""",8
…,…,…
"""gla""","""M""",10
"""lsdb""","""F""",13
"""lsdb""","""M""",13


In [7]:
# ---------- Special row: "No salary" default factor ----------
# Some AIDS salary factor tables have a "No salary" fallback row
# meaning: if the member's salary is missing/null, use this factor.
# We represent it with salary_from = None so pricing code can look it up
# via .filter(pl.col("salary_from").is_null())
print("\n🔎 Adding 'No salary' fallback rows...")

NO_SALARY_FALLBACKS = [
    # (product, gender, factor)
    ("ci_aids_acc", "both", 1.0),
    ("spouse_aids", "both", 1.0),
]

extra_rows = []
for product, gender, factor in NO_SALARY_FALLBACKS:
    extra_rows.append(dict(product=product, gender=gender,
                           salary_from=None, inc_adj=factor, acaf_adj=None))
    print(f"  ✅ {product:<15} {gender:<5} salary_from=None → factor={factor}")

df = pl.concat([df, pl.DataFrame(extra_rows)]) \
       .sort(["product", "gender", "salary_from"], nulls_last=True)

print(f"\n📊 Updated total rows: {df.height}")


🔎 Adding 'No salary' fallback rows...
  ✅ ci_aids_acc     both  salary_from=None → factor=1.0
  ✅ spouse_aids     both  salary_from=None → factor=1.0

📊 Updated total rows: 105


In [8]:
LEGACY_ADB_SALARY_MARKERS = [
    ("adb", "both", 0.0),
    ("adb", "both", 20534.0),
    ("adb", "both", 40962.0),
]

for product, gender, sf in LEGACY_ADB_SALARY_MARKERS:
    hit = df.filter(
        (pl.col("product") == product) &
        (pl.col("gender") == gender) &
        (pl.col("salary_from") == sf)
    )
    if not hit.is_empty():
        raise AssertionError(
            f"❌ Legacy adb salary factor detected: {gender} salary_from={sf}. "
            f"The source sheet may have been reverted to the old flat methodology."
        )

print("  ✅ No legacy adb rows detected")
SANITY_CHECKS = [
    # (product, gender, salary_from, expected_inc_adj, expected_acaf_or_None)
    ("gla",         "M",     0.0,      1.55,  0.95),
    ("gla",         "F", 70000.0,      1.35,  1.00),
    ("ci",          "both",  0.0,      1.10,  None),
    ("ci",          "both", 50123.0,   1.00,  None),
    ("ci_aids_acc", "both",  0.0,      1.60,  None),
    ("spouse",      "both",  0.0,      1.17,  None),
    ("spouse_aids", "both",  0.0,      1.60,  None),
    ("adb", "M", 0.0,      0.75,   None),
    ("adb", "M", 216000.0, 0.9375, None),
    ("adb", "M", 480000.0, 1.125,  None),
    ("adb", "F", 0.0,      0.75,   None),
    ("adb", "F", 204000.0, 0.9375, None),
    ("adb", "F", 420000.0, 1.125,  None),
    ("lsdb",        "M",     1.0,      0.70,  1.05),   # first band, salary_from=1
    ("lsdb",        "F", 940000.0,     0.65,  1.05),
]

print("🧪 Sanity checks:\n")
all_pass = True
for product, gender, sf, exp_inc, exp_acaf in SANITY_CHECKS:
    hit = df.filter(
        (pl.col("product") == product) &
        (pl.col("gender") == gender) &
        (pl.col("salary_from") == sf)
    )
    if hit.is_empty():
        print(f"  ❌ NOT FOUND  {product:<12} {gender:<5} salary_from={sf}")
        all_pass = False
        continue
    row = hit.row(0, named=True)
    inc_ok  = abs((row["inc_adj"] or 0) - exp_inc) < 1e-6
    acaf_ok = (exp_acaf is None) or (row["acaf_adj"] is not None
                                     and abs(row["acaf_adj"] - exp_acaf) < 1e-6)
    status = "✅ MATCH" if (inc_ok and acaf_ok) else "❌ MISMATCH"
    if not (inc_ok and acaf_ok):
        all_pass = False
    print(f"  {status}  {product:<12} {gender:<5} sf={sf:<9}  "
          f"inc={row['inc_adj']}  acaf={row['acaf_adj']}")

print(f"\n{'🎉 ALL PASSED' if all_pass else '⚠️ FIX FAILURES BEFORE WRITING'}")

  ✅ No legacy adb rows detected
🧪 Sanity checks:

  ✅ MATCH  gla          M     sf=0.0        inc=1.55  acaf=0.95
  ✅ MATCH  gla          F     sf=70000.0    inc=1.35  acaf=1.0
  ✅ MATCH  ci           both  sf=0.0        inc=1.1  acaf=None
  ✅ MATCH  ci           both  sf=50123.0    inc=1.0  acaf=None
  ✅ MATCH  ci_aids_acc  both  sf=0.0        inc=1.6  acaf=None
  ✅ MATCH  spouse       both  sf=0.0        inc=1.17  acaf=None
  ✅ MATCH  spouse_aids  both  sf=0.0        inc=1.6  acaf=None
  ✅ MATCH  adb          M     sf=0.0        inc=0.75  acaf=None
  ✅ MATCH  adb          M     sf=216000.0   inc=0.9375  acaf=None
  ✅ MATCH  adb          M     sf=480000.0   inc=1.125  acaf=None
  ✅ MATCH  adb          F     sf=0.0        inc=0.75  acaf=None
  ✅ MATCH  adb          F     sf=204000.0   inc=0.9375  acaf=None
  ✅ MATCH  adb          F     sf=420000.0   inc=1.125  acaf=None
  ✅ MATCH  lsdb         M     sf=1.0        inc=0.7  acaf=1.05
  ✅ MATCH  lsdb         F     sf=940000.0   inc=0.65  

In [9]:
# ---------- Guard: Excel file lock ----------
if MASTER_XLSX.exists():
    try:
        with open(MASTER_XLSX, "a"):
            pass
    except PermissionError:
        raise PermissionError(
            f"❌ '{MASTER_XLSX.name}' is open in Excel. Close it and re-run.")

# ---------- 1. Parquet snapshot ----------
parquet_path = SNAPSHOT_DIR / "salary_band.parquet"
df.write_parquet(parquet_path)
parquet_size = parquet_path.stat().st_size / 1024
parquet_hash = sha256_of(parquet_path)
print(f"📦 Parquet: {parquet_path.name}  ({parquet_size:.1f} KB)")

# ---------- 2. Master xlsx sheet ----------
if MASTER_XLSX.exists():
    wb_out = openpyxl.load_workbook(MASTER_XLSX)
    if "salary_band" in wb_out.sheetnames:
        del wb_out["salary_band"]
else:
    wb_out = openpyxl.Workbook()
    if "Sheet" in wb_out.sheetnames:
        del wb_out["Sheet"]

ws = wb_out.create_sheet("salary_band")
headers = ["product", "gender", "salary_from", "inc_adj", "acaf_adj"]
ws.append(headers)

# Style the header row
for cell in ws[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill("solid", fgColor="305496")
    cell.alignment = Alignment(horizontal="left")

for row in df.iter_rows():
    ws.append(row)

widths = {"A": 18, "B": 8, "C": 14, "D": 12, "E": 12}
for col, w in widths.items():
    ws.column_dimensions[col].width = w
ws.freeze_panes = "A2"

table = Table(displayName="tbl_salary_band", ref=f"A1:E{ws.max_row}")
table.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
ws.add_table(table)
wb_out.save(MASTER_XLSX)
print(f"📄 Master xlsx: sheet 'salary_band' → {df.height} rows")

# ---------- 3. Append build log row ----------
source_hash = sha256_of(SOURCE)
timestamp   = datetime.now(timezone.utc).isoformat(timespec="seconds")

log_row = {
    "timestamp_utc":     timestamp,
    "sheet":             "salary_band",
    "row_count":         df.height,
    "distinct_products": df["product"].n_unique(),
    "source_file":       SOURCE.name,
    "source_sha256":     source_hash,
    "parquet_file":      parquet_path.name,
    "parquet_sha256":    parquet_hash,
    "parquet_kb":        round(parquet_size, 2),
    "status":            "ok",
    "built_by":          "Mohamed Hussein Gadiwala",
}

log_exists = BUILD_LOG.exists()
with open(BUILD_LOG, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(log_row.keys()))
    if not log_exists:
        writer.writeheader()
    writer.writerow(log_row)

print(f"📝 Build log:  appended row to {BUILD_LOG.name}")
print(f"\n🎉 Done!")


📦 Parquet: salary_band.parquet  (2.5 KB)
📄 Master xlsx: sheet 'salary_band' → 105 rows
📝 Build log:  appended row to build_log.csv

🎉 Done!
